# 04.3 — Failures and AI API previews

[Class index and setup](README.md) · [Previous: 04.2](04.2-a-small-validated-client.ipynb) · [Next: 04.4](04.4-public-data-and-authentication.ipynb)

**Purpose:** Recognize which boundary failed, then recognize the same HTTP pattern in AI APIs.

**In class:** 30 minutes. **Independent work:** 15 minutes, included in the index's 210-minute allowance. Preparation has its own 30 minutes. The instructor's separate project is additional work.

## Learning outcomes

- Distinguish connection, timeout, HTTP, JSON, and record failures.
- Explain a useful error instead of converting failures to empty data.
- Identify POST, a JSON body, and an authentication header in AI API previews.

## Starting point

You know basic Python, functions, JSON-like dictionaries, and the earlier Observation/Report idea. The complete package and fixtures are supplied again; no previous notebook or exercise answer is needed. Open the supplied class workspace in VS Code, select this class's `.venv`, then **Restart Kernel → Run All**.

Default cells inspect saved responses and run offline checks. They do not start a server, call an external API, read credentials, install software, or download models. Live commands are explicit **manual terminal steps**. An HTTP-shaped fixture is not evidence of a live request. Empty answer cells are safe.

Attempt exercises before using the separate `%load` cell: uncomment its last line and run to reveal, then run the populated cell separately. A loaded solution replaces its loader cell; restarting the kernel does not restore that source. Keep your attempt in its own cell.

In [ ]:
from pathlib import Path
import sys

if not Path("report_app/api.py").is_file() or not Path("fixtures/observations.json").is_file():
    raise FileNotFoundError(
        "Open 04-consuming-apis.code-workspace and restart the kernel. "
        "Download the complete class folder, including report_app/, fixtures/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available; default execution is offline.")

## In class · 60–70 minutes — Failure is information

An API can fail before sending a response, return an error status, or return successful HTTP containing unusable data. Handle these at the boundary where they become meaningful.

| Observation | Layer | First useful action |
| --- | --- | --- |
| Cannot connect | Transport/service | Check the address and start the local server |
| Timeout | Transport/service | Check availability; try again later if appropriate |
| HTTP 401 | Access | Check authorized credentials; do not repeatedly resend the same invalid key |
| HTTP 404 | Resource | Check the endpoint path |
| HTTP 429 | Capacity | Stop sending requests; follow the service's rate-limit guidance |
| HTTP 503 | Service | Check service status and try later |
| HTTP 200 but HTML | Body format | Inspect the response format; this may be a proxy or login page |
| JSON but missing value | Data contract | Inspect the invalid record; missing does not mean zero |

These are diagnosis clues, not guarantees about every provider. The required client makes one attempt and reports the failure. Pagination and retries are optional extensions; do not add a loop just because something failed.

`raise ... from exc` preserves the original cause while explaining the application's problem. Catch only expected failures. A broad `except: return []` would turn an unavailable service into an apparently successful empty report.

In [ ]:
from unittest.mock import patch
import requests
from examples.saved_http import saved_response
from report_app.api import fetch_observations

cases = [("unavailable.json", 503), ("not-json.html", 200), ("invalid-record.json", 200)]
for name, status in cases:
    with patch("report_app.api.requests.get", return_value=saved_response(name, status)):
        try:
            fetch_observations()
        except (RuntimeError, ValueError) as exc:
            print(name, "→", type(exc).__name__, str(exc))
        else:
            raise AssertionError("Expected this teaching input to fail")

with patch("report_app.api.requests.get", side_effect=requests.Timeout()):
    try:
        fetch_observations()
    except RuntimeError as exc:
        print("Timeout →", exc)

## In class · 70–80 minutes — Two familiar AI services are APIs too

OpenAI and Ollama are short **instructor demonstrations**, not required student setup. Both examples use an HTTP `POST` with a JSON request body. This differs from the earlier GET query. The response is still inspected as status plus JSON, but each provider has its own body format.

| Demo | Address | Request | Access/configuration |
| --- | --- | --- | --- |
| OpenAI Responses | `https://api.openai.com/v1/responses` | `model` and `input` in JSON | Bearer key in a header; instructor chooses an accessible model |
| Local Ollama | `http://127.0.0.1:11434/api/generate` | `model`, `prompt`, `stream: false` | Already-running local server and an installed model |

Read [ai_http.py](examples/ai_http.py). The live branches show `requests.post(..., json=..., timeout=...)`; the OpenAI branch also supplies `headers=`. Setting `stream=False` asks Ollama for one complete JSON object. There is no requirement to install either provider's SDK here. Class 05 will teach the OpenAI SDK and bounded application logic.

The two fixtures below are **hand-authored illustrative envelopes, not captured live model answers**. Running them demonstrates data shape, not service availability or model quality.

In [ ]:
from examples.ai_http import openai_response, ollama_response

openai_payload = openai_response()
ollama_payload = ollama_response()
print("ILLUSTRATIVE OpenAI status:", openai_payload["status"])
print("ILLUSTRATIVE OpenAI output:", openai_payload["output"])
print("ILLUSTRATIVE Ollama text:", ollama_payload["response"])
print("ILLUSTRATIVE Ollama completed:", ollama_payload["done"])

### Optional live instructor switch — no student credentials needed

The exact commands and environment settings are in [AI_DEMOS.md](AI_DEMOS.md). The instructor can run either `python -m examples.ai_http openai --live` or `python -m examples.ai_http ollama --live` after preparation. Without `--live`, both commands read saved JSON and do not even read the credential/model environment variables.

The OpenAI key belongs in the instructor's terminal/server environment, never in a distributed notebook or frontend. A live call may incur charges. Ollama requires a local installation and a downloaded model; do not spend the required class time installing it. No particular model's CPU suitability is claimed here.

A 200 status does not establish that an answer is factually right. OpenAI output may include non-text items, a refusal, or incomplete results; inspect status/content instead of assuming the first output item is text. Ollama's response has a different envelope. Save application-level AI validation for class 05.

### 🧩 Exercise 04.3.E1 — Diagnose three different failures

> **In class · 10 minutes · 80–90**
>
> **Goal:** Say which boundary failed and give an actionable next step.
>
> **Start here:** The three saved cases and fetch_observations above.
>
> **Task:** For HTTP 503, HTTP 200 with HTML, and HTTP 200 with a missing value, predict the failure layer, run each controlled case, and record the message. Explain why neither [] nor value=0 is an acceptable automatic repair.
>
> **Check:** You identify status, JSON, and schema failures separately. All three reject the load, with useful explanations; none silently returns a partial summary.
>
> **Hint:** Check status before JSON. An HTTP success can still contain invalid data. Preserve the cause when adding an explanation.

In [ ]:
# Your attempt for 04.3.E1.
# Record code, observations, and your explanation here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_04_3_e1.py

## Guided follow-up — Independent work · 15 minutes

Spend five minutes reading the small failure branches in [api.py](report_app/api.py), five classifying timeout/401/404/429 examples using the table, and five writing an explanation of a useful message for a peer. The OpenAI/Ollama replay remains available without setup; do not count optional installation or model downloads as required work.

**Completion:** explain what the user can do next and which original cause would help a developer debug. Avoid printing authorization headers while investigating a live request.

## References

- [Requests errors and timeouts](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [Official OpenAI Responses API](https://developers.openai.com/api/reference/resources/responses/methods/create)
- [Ollama generate endpoint](https://docs.ollama.com/api/generate)
- [Class index](README.md)